# Trajectory Flow Matching: minimal Colab reproduction

This is a thin launcher for the repository's existing `src/main.py`. It does not duplicate TFM-SDE training logic. Before running, select **Runtime → Change runtime type**, choose runtime version **2025.07** (Python 3.11 / PyTorch 2.6.0), and select a GPU hardware accelerator when desired.

## 1. Clone or open the repository

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/nZhangx/TrajectoryFlowMatching.git"
REPO_DIR = Path("/content/TrajectoryFlowMatching")

if not (REPO_DIR / "src" / "main.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Using existing repository at {REPO_DIR}")

os.chdir(REPO_DIR)
print("Repository:", Path.cwd())
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 2. Verify Python and install dependencies

PyTorch is deliberately not installed by `requirements-colab.txt`; this retains Colab's CUDA-matched PyTorch build.

In [ ]:
print("Python:", sys.version)
if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "This launcher targets Colab runtime version 2025.07 with Python 3.11. "
        "Select that past runtime before installing dependencies."
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
print("Dependencies installed. If Colab requests a restart, restart and rerun from cell 1.")

## 3. Verify hardware and important imports

In [ ]:
import importlib
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

important_modules = [
    "pytorch_lightning", "hydra", "omegaconf", "torchdyn",
    "torchdiffeq", "torchsde", "ot", "pandas", "wandb",
]
for module_name in important_modules:
    module = importlib.import_module(module_name)
    print(f"{module_name}: {getattr(module, '__version__', 'imported')}")

src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from data.datamodule import clinical_DataModule
from model.mlp_noise import Noise_MLP_Cond_Memory_Module
print("clinical_DataModule import: OK")
print("Noise_MLP_Cond_Memory_Module import: OK")

## 4. Run a one-batch smoke test through `src/main.py`

This uses the bundled eICU configuration, clinical DataModule, TFM-SDE model, Lightning Trainer, validation sanity path, and test path. Hydra overrides limit each loop to one batch and disable WandB. The model remains `tfm_sde` with memory 3.

In [ ]:
smoke_command = [
    sys.executable, "src/main.py",
    "data=eICU",
    "model=tfm_sde",
    "wandb_logging=false",
    "max_epochs=1",
    "max_time=00:00:10:00",
    "limit_train_batches=1",
    "limit_val_batches=1",
    "limit_test_batches=1",
    "num_sanity_val_steps=1",
    "hydra.run.dir=.",
]
print("Running:", " ".join(smoke_command))
subprocess.run(smoke_command, cwd=REPO_DIR, check=True)

## Full baseline command (not run here)

```bash
python src/main.py data=eICU model=tfm_sde wandb_logging=false hydra.run.dir=.
```